In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Tuple

def simulate_demo_data(n: int = 100, outlier_fraction: float = 0.05) -> pd.DataFrame:
    """Simulate demo data with a specified fraction of outliers.

    Args:
        n (int): Number of normal data points.
        outlier_fraction (float): Fraction of outliers to add.

    Returns:
        pd.DataFrame: DataFrame containing the simulated data.
    """
    np.random.seed(42)
    data: np.ndarray = np.random.normal(loc=50, scale=10, size=n)
    n_outliers: int = int(n * outlier_fraction)
    outliers: np.ndarray = np.random.uniform(low=100, high=150, size=n_outliers)
    data_with_outliers: np.ndarray = np.concatenate([data, outliers])
    return pd.DataFrame({'value': data_with_outliers})

def detect_outliers_iqr(df: pd.DataFrame, column: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Detect outliers in a DataFrame column using the IQR method.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Column name to check for outliers.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: DataFrames of non-outliers and outliers.
    """
    q1: float = df[column].quantile(0.25)
    q3: float = df[column].quantile(0.75)
    iqr: float = q3 - q1
    lower_bound: float = q1 - 1.5 * iqr
    upper_bound: float = q3 + 1.5 * iqr
    outliers: pd.DataFrame = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    non_outliers: pd.DataFrame = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return non_outliers, outliers

def plot_box_with_outliers(df: pd.DataFrame, column: str) -> None:
    """Plot a professional boxplot for a DataFrame column, highlighting outliers.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Column name to plot.
    """
    plt.figure(figsize=(10, 6))
    box = plt.boxplot(
        df[column],
        vert=False,
        patch_artist=True,
        boxprops=dict(facecolor='#4F81BD', color='#2F4F4F', linewidth=2),
        whiskerprops=dict(color='#2F4F4F', linewidth=2),
        capprops=dict(color='#2F4F4F', linewidth=2),
        medianprops=dict(color='#C0504D', linewidth=2),
        flierprops=dict(marker='o', markerfacecolor='#C0504D', markersize=8, linestyle='none', markeredgecolor='#2F4F4F')
    )
    plt.title('Boxplot of Value with Outlier Detection', fontsize=16, fontweight='bold')
    plt.xlabel(column, fontsize=14)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tick_params(axis='both', which='major', labelsize=12)
    plt.tight_layout()
    plt.show()

def create_stats_table(df: pd.DataFrame, outliers_df: pd.DataFrame, column: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Create statistics and outlier tables for a DataFrame column.

    Args:
        df (pd.DataFrame): Input DataFrame.
        outliers_df (pd.DataFrame): DataFrame of outliers.
        column (str): Column name to compute statistics.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Statistics table and outlier values table.
    """
    stats: dict = {
        'mean': df[column].mean(),
        'median': df[column].median(),
        'q1': df[column].quantile(0.25),
        'q3': df[column].quantile(0.75),
        'iqr': df[column].quantile(0.75) - df[column].quantile(0.25),
        'min': df[column].min(),
        'max': df[column].max(),
        'num_outliers': len(outliers_df),
        'outliers': outliers_df[column].values
    }
    stats_df: pd.DataFrame = pd.DataFrame({
        'Statistic': ['Mean', 'Median', 'Q1', 'Q3', 'IQR', 'Min', 'Max', 'Num Outliers'],
        'Value': [stats['mean'], stats['median'], stats['q1'], stats['q3'], stats['iqr'], stats['min'], stats['max'], stats['num_outliers']]
    })
    outliers_table: pd.DataFrame = pd.DataFrame({'Outlier Values': stats['outliers']})
    return stats_df, outliers_table

# Simulate data
demo_df: pd.DataFrame = simulate_demo_data()

# Detect outliers
non_outliers_df, outliers_df = detect_outliers_iqr(demo_df, 'value')

# Plot boxplot
plot_box_with_outliers(demo_df, 'value')

# Create and display stats table and outliers table
stats_df, outliers_table = create_stats_table(demo_df, outliers_df, 'value')
display(stats_df)
display(outliers_table)